# P1-26b · Final mini-MIAS mask orientation, re-scoring and contrasts (CPU, ~15 min)
Run after P1-26. `FINAL` (cell 1) holds the orientation of every mask whose orientation was checked visually on the overlay sheets
(`revision_checks/mias_sheets/`); all other masks keep the orientation chosen by the image-only criterion of P1-26.
Writes only `MyDrive/P1_v2/revision_checks/`.

In [ ]:
# 1 · Drive, paths, visually verified orientations
from google.colab import drive
drive.mount('/content/drive', force_remount=True, timeout_ms=300000)
import os, sys, glob, json, re, io
import numpy as np, pandas as pd, cv2, requests
ROOT = '/content/drive/MyDrive/QSEG_FINAL'; OUT = '/content/drive/MyDrive/P1_v2'; MIAS = f'{OUT}/mias_ext'; RC = f'{OUT}/revision_checks'
sys.path.insert(0, ROOT + '/code')
from qseg.data_prep import pad_to_square
FINAL = json.load(open(f'{RC}/orientation_visual.json'))      # {file_id: transform} for the visually checked masks
OR = pd.read_csv(f'{RC}/mias_orientation.csv')
OR['final'] = [FINAL.get(f, io_) for f, io_ in zip(OR.file_id, OR.image_only)]
OR.to_csv(f'{RC}/mias_orientation_final.csv', index=False)
print('visually set:', FINAL); print('final differs from the paper for:', OR[OR.final != OR.used_in_paper].file_id.tolist())

In [ ]:
# 2 · Masks in the final orientation (same padding and resizing as P1-16c)
!pip -q install pynrrd
import nrrd, zipfile, tarfile, shutil
MR = '/content/mias_raw'; os.makedirs(MR, exist_ok=True)
if not glob.glob(f'{MR}/masks/**/*.nrrd', recursive=True):
    rec = requests.get('https://zenodo.org/api/records/10149914', timeout=60).json()
    for f in rec['files']:
        url = f['links'].get('self') or f['links'].get('download'); dst = f"{MR}/{f['key']}"
        if not os.path.exists(dst): open(dst, 'wb').write(requests.get(url, timeout=600).content)
        if dst.endswith('.zip'): zipfile.ZipFile(dst).extractall(f'{MR}/masks')
mask_by_id = {}
for p in glob.glob(f'{MR}/masks/**/*.nrrd', recursive=True):
    m = re.search(r'mdb(\d{3})', os.path.basename(p).lower())
    if m: mask_by_id['mdb' + m.group(1)] = p
T = {'id': lambda a: a, 'T': lambda a: a.T, 'ud': lambda a: a[::-1], 'Tud': lambda a: a.T[::-1], 'lr': lambda a: a[:, ::-1], 'Tlr': lambda a: a.T[:, ::-1]}
def to256(m):
    return cv2.resize(pad_to_square(m.astype(np.uint8) * 255, 'center'), (256, 256), interpolation=cv2.INTER_NEAREST) > 127
NEW = {}
for _, r in OR.iterrows():
    lab, _ = nrrd.read(mask_by_id[r.file_id]); lab = np.ascontiguousarray(T[r.final](np.squeeze(lab)))
    NEW[r.file_id] = dict(full=to256((lab == 1) | (lab == 2)), breast=to256(lab == 1))
os.makedirs(f'{RC}/masks_final/full', exist_ok=True); os.makedirs(f'{RC}/masks_final/breast', exist_ok=True)
for f, d in NEW.items():
    cv2.imwrite(f'{RC}/masks_final/full/{f}.png', d['full'].astype(np.uint8) * 255); cv2.imwrite(f'{RC}/masks_final/breast/{f}.png', d['breast'].astype(np.uint8) * 255)
same = sum(bool(((cv2.imread(f'{MIAS}/masks_expert/{f}.png', 0) > 127) == d['full']).all()) for f, d in NEW.items())
print('masks identical to those used in the paper:', same, 'of', len(NEW))

In [ ]:
# 3 · Re-score all saved runs and compute the circuit contrasts (crossed bootstrap over women and seeds, 10,000 replicates, rng(0))
def dice(a, b):
    s = a.sum() + b.sum(); return 1.0 if s == 0 else 2 * (a & b).sum() / s
rows = []
for d in sorted(glob.glob(f'{OUT}/runs/*_seed*')):
    f = f'{d}/mias_prob_uint8.npz'
    if not os.path.exists(f): continue
    z = np.load(f, allow_pickle=True); P = z['prob'] >= 128; fids = [str(x) for x in z['file_id']]
    exp, mdl, s = os.path.basename(d).split('_')
    seen = set()
    for k, fid in enumerate(fids):
        if fid in seen: continue
        seen.add(fid)
        rows.append(dict(exp=exp, model=mdl, seed=int(s[4:]), file_id=fid, dice_full=dice(P[k], NEW[fid]['full']), dice_breast=dice(P[k], NEW[fid]['breast'])))
S = pd.DataFrame(rows); S.to_csv(f'{RC}/mias_final_per_image.csv.gz', index=False)
S['cluster'] = ((S.file_id.str[3:].astype(int) + 1) // 2).astype(str)
B = 10000
def boot(u, rng):
    seeds = np.sort(u.seed.unique()); cl = np.sort(u.cluster.unique())
    ci = {c: i for i, c in enumerate(cl)}; si = {s: i for i, s in enumerate(seeds)}
    Sm = np.zeros((len(cl), len(seeds))); N = np.zeros((len(cl), len(seeds)))
    for c, s, v in zip(u.cluster, u.seed, u.v): Sm[ci[c], si[s]] += v; N[ci[c], si[s]] += 1
    wc = rng.multinomial(len(cl), np.ones(len(cl)) / len(cl), size=B); ws = rng.multinomial(len(seeds), np.ones(len(seeds)) / len(seeds), size=B)
    bs = np.einsum('bc,cs,bs->b', wc, Sm, ws) / np.einsum('bc,cs,bs->b', wc, N, ws)
    return Sm.sum() / N.sum(), np.percentile(bs, 2.5), np.percentile(bs, 97.5), max(min(1.0, 2 * min((bs <= 0).mean(), (bs >= 0).mean())), 1 / B)
out = []
for col in ('dice_full', 'dice_breast'):
    for exp in ('E1', 'E2'):
        t = S[S.exp == exp]
        for m in ('hq', 'hc', 'hf', 'ht'):
            u = t[t.model == m].rename(columns={col: 'v'}); e, lo, hi, p = boot(u, np.random.default_rng(0))
            out.append(dict(metric=col, exp=exp, contrast=m, est=e, lo=lo, hi=hi, p=p))
        for b in ('hc', 'hf', 'ht'):
            x = t[t.model == 'hq'][['file_id', 'cluster', 'seed', col]].rename(columns={col: 'a'})
            y = t[t.model == b][['file_id', 'cluster', 'seed', col]].rename(columns={col: 'b'})
            mm_ = x.merge(y, on=['file_id', 'cluster', 'seed']); mm_['v'] = mm_.a - mm_.b
            e, lo, hi, p = boot(mm_, np.random.default_rng(0)); out.append(dict(metric=col, exp=exp, contrast=f'hq-{b}', est=e, lo=lo, hi=hi, p=p))
C = pd.DataFrame(out); C.to_csv(f'{RC}/mias_final_contrasts.csv', index=False)
pd.set_option('display.width', 200); print(C.round(4).to_string(index=False))

In [ ]:
# 4 · DMID metadata: is there a patient identifier?
art = requests.get('https://api.figshare.com/v2/articles/24522883', timeout=60).json()
f = [x for x in art['files'] if x['name'] == 'Metadata.xlsx'][0]
X = pd.read_excel(io.BytesIO(requests.get(f['download_url'], timeout=120).content), header=None)
X.to_csv(f'{RC}/dmid_metadata_raw.csv', index=False)
pd.set_option('display.max_columns', 20); pd.set_option('display.width', 250)
print(X.head(70).to_string()); print('...'); print(X.tail(15).to_string())
print('DONE')